# Federal Reserve Data Cleansing Pipeline

This notebook provides a comprehensive data cleansing pipeline for Federal Reserve Economic Data (FRED) used in our policy analysis.

## Overview
- **Purpose**: Clean and prepare FRED datasets for analysis
- **Data Sources**: FRED API, SQLite database, CSV files
- **Output**: Clean, standardized datasets ready for analysis

## Datasets Covered
1. Federal Funds Rate (FEDFUNDS)
2. Inflation Data (CPI, PCE)
3. Employment Data (UNRATE, PAYEMS)
4. Balance Sheet Data (WALCL)
5. Bond Yields (GS10, GS2)
6. Economic Indicators (GDP, INDPRO)

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import sqlite3
import requests
import os
import warnings
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Optional, Tuple, Union
import logging

# Configure warnings and logging
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("📊 Federal Reserve Data Cleansing Pipeline Initialized")
print(f"📅 Run Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 1. Data Cleansing Configuration

In [ ]:
class FedDataCleansingConfig:
    """Configuration class for data cleansing parameters"""
    
    # FRED API Configuration
    FRED_API_BASE_URL = "https://api.stlouisfed.org/fred/series/observations"
    FRED_API_KEY = os.getenv('FRED_API_KEY', '852567bee469ec614405e5dce75b847d')
    
    # Database Configuration
    DATABASE_PATH = 'inflation_data.db'
    
    # Data Quality Thresholds
    MAX_MISSING_PERCENTAGE = 0.1  # 10% maximum missing data
    OUTLIER_Z_SCORE_THRESHOLD = 3.0
    MIN_DATA_POINTS = 12  # Minimum 12 data points for series
    
    # Date Range Configuration
    DEFAULT_START_DATE = '2000-01-01'
    DEFAULT_END_DATE = datetime.now().strftime('%Y-%m-%d')
    
    # FRED Series IDs for Federal Reserve Analysis
    FRED_SERIES = {
        'rates': {
            'FEDFUNDS': 'Federal Funds Rate',
            'GS10': '10-Year Treasury Rate',
            'GS2': '2-Year Treasury Rate',
            'GS5': '5-Year Treasury Rate',
            'TB3MS': '3-Month Treasury Rate'
        },
        'inflation': {
            'CPIAUCSL': 'Consumer Price Index',
            'CPILFESL': 'Core CPI',
            'PCEPI': 'PCE Price Index',
            'PCEPILFE': 'Core PCE Price Index'
        },
        'employment': {
            'UNRATE': 'Unemployment Rate',
            'PAYEMS': 'Nonfarm Payrolls',
            'CIVPART': 'Labor Force Participation Rate',
            'EMRATIO': 'Employment-Population Ratio'
        },
        'monetary_policy': {
            'WALCL': 'Fed Balance Sheet',
            'BOGMBASE': 'Monetary Base',
            'M2SL': 'M2 Money Supply'
        },
        'economic_activity': {
            'GDP': 'Gross Domestic Product',
            'INDPRO': 'Industrial Production Index',
            'HOUST': 'Housing Starts',
            'UMCSENT': 'Consumer Sentiment'
        }
    }

config = FedDataCleansingConfig()
print("✅ Configuration loaded successfully")
print(f"📊 Total series to process: {sum(len(category) for category in config.FRED_SERIES.values())}")

## 2. Data Quality Assessment Functions

In [ ]:
class DataQualityAssessment:
    """Class for assessing data quality issues"""
    
    @staticmethod
    def assess_missing_data(df: pd.DataFrame, series_name: str) -> Dict:
        """Assess missing data in a series"""
        total_records = len(df)
        missing_records = df.isnull().sum().sum() if hasattr(df.isnull().sum(), 'sum') else df.isnull().sum()
        missing_percentage = (missing_records / total_records) * 100 if total_records > 0 else 0
        
        return {
            'series_name': series_name,
            'total_records': total_records,
            'missing_records': missing_records,
            'missing_percentage': missing_percentage,
            'quality_flag': 'GOOD' if missing_percentage <= config.MAX_MISSING_PERCENTAGE * 100 else 'POOR'
        }
    
    @staticmethod
    def detect_outliers(series: pd.Series, method: str = 'zscore') -> pd.Series:
        """Detect outliers using specified method"""
        if method == 'zscore':
            z_scores = np.abs((series - series.mean()) / series.std())
            return z_scores > config.OUTLIER_Z_SCORE_THRESHOLD
        elif method == 'iqr':
            Q1 = series.quantile(0.25)
            Q3 = series.quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            return (series < lower_bound) | (series > upper_bound)
        else:
            raise ValueError("Method must be 'zscore' or 'iqr'")
    
    @staticmethod
    def assess_data_consistency(df: pd.DataFrame, date_column: str = 'date') -> Dict:
        """Assess data consistency and continuity"""
        if date_column not in df.columns:
            return {'error': f'Date column {date_column} not found'}
        
        df_sorted = df.sort_values(date_column)
        date_diff = df_sorted[date_column].diff()
        
        # Remove first NaT value from diff
        date_diff = date_diff.dropna()
        
        if len(date_diff) == 0:
            return {'error': 'Insufficient data for consistency assessment'}
        
        # Calculate frequency statistics
        most_common_freq = date_diff.mode().iloc[0] if len(date_diff.mode()) > 0 else None
        
        return {
            'date_range': f"{df_sorted[date_column].min()} to {df_sorted[date_column].max()}",
            'total_periods': len(df_sorted),
            'most_common_frequency': most_common_freq,
            'irregular_intervals': len(date_diff[date_diff != most_common_freq]) if most_common_freq else 0,
            'data_continuity': 'GOOD' if most_common_freq else 'POOR'
        }

quality_assessor = DataQualityAssessment()
print("✅ Data Quality Assessment functions initialized")

## 3. Data Loading Functions

In [ ]:
class FedDataLoader:
    """Class for loading Federal Reserve data from various sources"""
    
    def __init__(self, api_key: str = None):
        self.api_key = api_key or config.FRED_API_KEY
        self.base_url = config.FRED_API_BASE_URL
    
    def fetch_fred_series(self, series_id: str, start_date: str = None, end_date: str = None) -> pd.DataFrame:
        """Fetch data from FRED API"""
        try:
            params = {
                'series_id': series_id,
                'api_key': self.api_key,
                'file_type': 'json',
                'observation_start': start_date or config.DEFAULT_START_DATE,
                'observation_end': end_date or config.DEFAULT_END_DATE
            }
            
            response = requests.get(self.base_url, params=params, timeout=30)
            response.raise_for_status()
            
            data = response.json()
            
            if 'observations' not in data:
                logger.warning(f"No observations found for series {series_id}")
                return pd.DataFrame()
            
            df = pd.DataFrame(data['observations'])
            
            if df.empty:
                logger.warning(f"Empty dataset returned for series {series_id}")
                return df
            
            # Basic preprocessing
            df['date'] = pd.to_datetime(df['date'])
            df[series_id] = pd.to_numeric(df['value'], errors='coerce')
            df = df[['date', series_id]].dropna()
            
            logger.info(f"Successfully fetched {len(df)} records for {series_id}")
            return df
            
        except Exception as e:
            logger.error(f"Error fetching {series_id}: {str(e)}")
            return pd.DataFrame()
    
    def load_from_database(self, table_name: str = 'fed_policy_data') -> pd.DataFrame:
        """Load data from SQLite database"""
        try:
            conn = sqlite3.connect(config.DATABASE_PATH)
            
            # Check if table exists
            cursor = conn.cursor()
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name=?", (table_name,))
            
            if not cursor.fetchone():
                logger.warning(f"Table {table_name} not found in database")
                return pd.DataFrame()
            
            df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
            conn.close()
            
            if 'date' in df.columns:
                df['date'] = pd.to_datetime(df['date'])
            
            logger.info(f"Loaded {len(df)} records from database table {table_name}")
            return df
            
        except Exception as e:
            logger.error(f"Error loading from database: {str(e)}")
            return pd.DataFrame()
    
    def load_from_csv(self, file_path: str) -> pd.DataFrame:
        """Load data from CSV file"""
        try:
            if not os.path.exists(file_path):
                logger.warning(f"CSV file not found: {file_path}")
                return pd.DataFrame()
            
            df = pd.read_csv(file_path)
            
            # Try to identify date column
            date_columns = ['date', 'Date', 'DATE', 'observation_date']
            for col in date_columns:
                if col in df.columns:
                    df[col] = pd.to_datetime(df[col], errors='coerce')
                    if col != 'date':
                        df = df.rename(columns={col: 'date'})
                    break
            
            logger.info(f"Loaded {len(df)} records from CSV: {file_path}")
            return df
            
        except Exception as e:
            logger.error(f"Error loading CSV {file_path}: {str(e)}")
            return pd.DataFrame()

data_loader = FedDataLoader()
print("✅ Data Loader initialized")

## 4. Data Cleansing Functions

In [ ]:
class FedDataCleaner:
    """Comprehensive data cleansing class for Federal Reserve data"""
    
    def __init__(self):
        self.cleaning_log = []
    
    def log_cleaning_action(self, action: str, series_name: str, details: str = ""):
        """Log cleaning actions for audit trail"""
        self.cleaning_log.append({
            'timestamp': datetime.now(),
            'action': action,
            'series': series_name,
            'details': details
        })
    
    def standardize_date_format(self, df: pd.DataFrame, date_column: str = 'date') -> pd.DataFrame:
        """Standardize date formats across datasets"""
        if date_column not in df.columns:
            logger.warning(f"Date column '{date_column}' not found")
            return df
        
        try:
            original_format = df[date_column].dtype
            df[date_column] = pd.to_datetime(df[date_column], errors='coerce')
            
            # Remove rows with invalid dates
            before_count = len(df)
            df = df.dropna(subset=[date_column])
            after_count = len(df)
            
            if before_count != after_count:
                self.log_cleaning_action(
                    'DATE_STANDARDIZATION', 
                    'UNKNOWN',
                    f"Removed {before_count - after_count} rows with invalid dates"
                )
            
            logger.info(f"Standardized date format from {original_format} to datetime64")
            return df
            
        except Exception as e:
            logger.error(f"Error standardizing dates: {str(e)}")
            return df
    
    def handle_missing_values(self, df: pd.DataFrame, series_name: str, method: str = 'interpolate') -> pd.DataFrame:
        """Handle missing values using specified method"""
        if df.empty:
            return df
        
        numeric_columns = df.select_dtypes(include=[np.number]).columns
        
        for col in numeric_columns:
            missing_count = df[col].isnull().sum()
            
            if missing_count == 0:
                continue
            
            original_count = len(df)
            
            if method == 'interpolate':
                df[col] = df[col].interpolate(method='linear')
            elif method == 'forward_fill':
                df[col] = df[col].fillna(method='ffill')
            elif method == 'backward_fill':
                df[col] = df[col].fillna(method='bfill')
            elif method == 'drop':
                df = df.dropna(subset=[col])
            elif method == 'mean':
                df[col] = df[col].fillna(df[col].mean())
            elif method == 'median':
                df[col] = df[col].fillna(df[col].median())
            
            self.log_cleaning_action(
                'MISSING_VALUES', 
                series_name,
                f"Column {col}: {missing_count} missing values handled using {method}"
            )
        
        return df
    
    def remove_outliers(self, df: pd.DataFrame, series_name: str, method: str = 'zscore', 
                       threshold: float = None) -> pd.DataFrame:
        """Remove or cap outliers"""
        if df.empty:
            return df
        
        threshold = threshold or config.OUTLIER_Z_SCORE_THRESHOLD
        numeric_columns = df.select_dtypes(include=[np.number]).columns
        
        for col in numeric_columns:
            if col == 'date':  # Skip date columns
                continue
                
            outliers = quality_assessor.detect_outliers(df[col], method=method)
            outlier_count = outliers.sum()
            
            if outlier_count > 0:
                # Cap outliers instead of removing them to preserve time series continuity
                if method == 'zscore':
                    mean_val = df[col].mean()
                    std_val = df[col].std()
                    upper_bound = mean_val + threshold * std_val
                    lower_bound = mean_val - threshold * std_val
                    
                    df.loc[df[col] > upper_bound, col] = upper_bound
                    df.loc[df[col] < lower_bound, col] = lower_bound
                
                elif method == 'iqr':
                    Q1 = df[col].quantile(0.25)
                    Q3 = df[col].quantile(0.75)
                    IQR = Q3 - Q1
                    upper_bound = Q3 + 1.5 * IQR
                    lower_bound = Q1 - 1.5 * IQR
                    
                    df.loc[df[col] > upper_bound, col] = upper_bound
                    df.loc[df[col] < lower_bound, col] = lower_bound
                
                self.log_cleaning_action(
                    'OUTLIER_TREATMENT', 
                    series_name,
                    f"Column {col}: {outlier_count} outliers capped using {method} method"
                )
        
        return df
    
    def standardize_frequency(self, df: pd.DataFrame, series_name: str, 
                            target_freq: str = 'M', date_column: str = 'date') -> pd.DataFrame:
        """Standardize data frequency (e.g., monthly, quarterly)"""
        if df.empty or date_column not in df.columns:
            return df
        
        try:
            df = df.sort_values(date_column)
            df = df.set_index(date_column)
            
            # Resample to target frequency
            if target_freq == 'M':  # Monthly
                df_resampled = df.resample('M').last()  # Use last value of month
            elif target_freq == 'Q':  # Quarterly
                df_resampled = df.resample('Q').last()
            elif target_freq == 'A':  # Annual
                df_resampled = df.resample('A').last()
            else:
                logger.warning(f"Unsupported frequency: {target_freq}")
                return df.reset_index()
            
            df_resampled = df_resampled.reset_index()
            
            self.log_cleaning_action(
                'FREQUENCY_STANDARDIZATION', 
                series_name,
                f"Resampled to {target_freq} frequency: {len(df)} -> {len(df_resampled)} records"
            )
            
            return df_resampled
            
        except Exception as e:
            logger.error(f"Error standardizing frequency for {series_name}: {str(e)}")
            return df.reset_index() if date_column in df.index.names else df
    
    def validate_data_range(self, df: pd.DataFrame, series_name: str, 
                          expected_min: float = None, expected_max: float = None) -> pd.DataFrame:
        """Validate data is within expected ranges"""
        if df.empty:
            return df
        
        numeric_columns = df.select_dtypes(include=[np.number]).columns
        
        for col in numeric_columns:
            if col == 'date':  # Skip date columns
                continue
            
            # Define reasonable ranges for common Fed series
            if expected_min is None or expected_max is None:
                if 'FEDFUNDS' in col or 'GS' in col or 'TB' in col:
                    # Interest rates: -1% to 25%
                    min_val, max_val = -1, 25
                elif 'UNRATE' in col:
                    # Unemployment rate: 0% to 25%
                    min_val, max_val = 0, 25
                elif 'CPI' in col and 'rate' in series_name.lower():
                    # Inflation rate: -10% to 20%
                    min_val, max_val = -10, 20
                else:
                    # Default: no range validation
                    continue
            else:
                min_val, max_val = expected_min, expected_max
            
            # Count and flag out-of-range values
            out_of_range = (df[col] < min_val) | (df[col] > max_val)
            out_of_range_count = out_of_range.sum()
            
            if out_of_range_count > 0:
                self.log_cleaning_action(
                    'RANGE_VALIDATION', 
                    series_name,
                    f"Column {col}: {out_of_range_count} values outside expected range [{min_val}, {max_val}]"
                )
                
                # Optionally cap extreme values
                df.loc[df[col] < min_val, col] = min_val
                df.loc[df[col] > max_val, col] = max_val
        
        return df
    
    def get_cleaning_summary(self) -> pd.DataFrame:
        """Get summary of all cleaning actions performed"""
        if not self.cleaning_log:
            return pd.DataFrame()
        
        return pd.DataFrame(self.cleaning_log)

data_cleaner = FedDataCleaner()
print("✅ Data Cleaner initialized")

## 5. Comprehensive Cleansing Pipeline

In [ ]:
class FedDataCleansingPipeline:
    """Main pipeline class that orchestrates the entire cleansing process"""
    
    def __init__(self):
        self.loader = FedDataLoader()
        self.cleaner = FedDataCleaner()
        self.quality_assessor = DataQualityAssessment()
        self.cleaned_datasets = {}
        self.quality_reports = {}
    
    def process_single_series(self, series_id: str, series_name: str, 
                            cleansing_options: Dict = None) -> Dict:
        """Process a single FRED series through the complete pipeline"""
        
        default_options = {
            'missing_value_method': 'interpolate',
            'outlier_method': 'zscore',
            'outlier_threshold': 3.0,
            'target_frequency': 'M',
            'validate_range': True
        }
        
        options = {**default_options, **(cleansing_options or {})}
        
        logger.info(f"Processing series: {series_id} ({series_name})")
        
        # Step 1: Load data
        df = self.loader.fetch_fred_series(series_id)
        
        if df.empty:
            logger.warning(f"No data loaded for {series_id}")
            return {'status': 'failed', 'reason': 'no_data'}
        
        # Step 2: Initial quality assessment
        initial_quality = self.quality_assessor.assess_missing_data(df, series_name)
        consistency_check = self.quality_assessor.assess_data_consistency(df)
        
        # Step 3: Data cleansing steps
        df = self.cleaner.standardize_date_format(df)
        df = self.cleaner.handle_missing_values(df, series_name, options['missing_value_method'])
        df = self.cleaner.remove_outliers(df, series_name, options['outlier_method'], options['outlier_threshold'])
        
        if options['validate_range']:
            df = self.cleaner.validate_data_range(df, series_name)
        
        df = self.cleaner.standardize_frequency(df, series_name, options['target_frequency'])
        
        # Step 4: Final quality assessment
        final_quality = self.quality_assessor.assess_missing_data(df, series_name)
        
        # Step 5: Store results
        self.cleaned_datasets[series_id] = df
        self.quality_reports[series_id] = {
            'series_name': series_name,
            'initial_quality': initial_quality,
            'final_quality': final_quality,
            'consistency_check': consistency_check,
            'cleansing_options': options,
            'final_record_count': len(df),
            'date_range': f"{df['date'].min()} to {df['date'].max()}" if not df.empty else "N/A"
        }
        
        logger.info(f"Successfully processed {series_id}: {len(df)} records")
        return {'status': 'success', 'records': len(df)}
    
    def process_all_series(self, categories: List[str] = None) -> Dict:
        """Process all series in specified categories"""
        
        categories = categories or list(config.FRED_SERIES.keys())
        results = {'success': [], 'failed': []}
        
        print(f"\n🚀 Starting comprehensive cleansing pipeline for {len(categories)} categories")
        print("=" * 70)
        
        for category in categories:
            if category not in config.FRED_SERIES:
                logger.warning(f"Category {category} not found in configuration")
                continue
            
            print(f"\n📊 Processing category: {category.upper()}")
            print("-" * 50)
            
            for series_id, series_name in config.FRED_SERIES[category].items():
                result = self.process_single_series(series_id, series_name)
                
                if result['status'] == 'success':
                    print(f"✅ {series_id}: {result['records']} records")
                    results['success'].append(series_id)
                else:
                    print(f"❌ {series_id}: {result.get('reason', 'unknown error')}")
                    results['failed'].append(series_id)
        
        print(f"\n📈 Pipeline Summary:")
        print(f"✅ Successful: {len(results['success'])} series")
        print(f"❌ Failed: {len(results['failed'])} series")
        
        return results
    
    def generate_quality_report(self) -> pd.DataFrame:
        """Generate comprehensive data quality report"""
        
        report_data = []
        
        for series_id, report in self.quality_reports.items():
            report_data.append({
                'Series_ID': series_id,
                'Series_Name': report['series_name'],
                'Initial_Records': report['initial_quality']['total_records'],
                'Final_Records': report['final_record_count'],
                'Initial_Missing_%': report['initial_quality']['missing_percentage'],
                'Final_Missing_%': report['final_quality']['missing_percentage'],
                'Quality_Flag': report['final_quality']['quality_flag'],
                'Date_Range': report['date_range'],
                'Data_Continuity': report['consistency_check'].get('data_continuity', 'UNKNOWN')
            })
        
        return pd.DataFrame(report_data)
    
    def export_cleaned_data(self, output_format: str = 'csv', output_dir: str = 'cleaned_data') -> None:
        """Export cleaned datasets"""
        
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        
        for series_id, df in self.cleaned_datasets.items():
            if output_format == 'csv':
                output_path = os.path.join(output_dir, f"{series_id}_cleaned.csv")
                df.to_csv(output_path, index=False)
            elif output_format == 'sqlite':
                output_path = os.path.join(output_dir, 'cleaned_fed_data.db')
                conn = sqlite3.connect(output_path)
                df.to_sql(f"{series_id}_cleaned", conn, if_exists='replace', index=False)
                conn.close()
        
        logger.info(f"Exported {len(self.cleaned_datasets)} datasets to {output_dir}")

# Initialize the main pipeline
pipeline = FedDataCleansingPipeline()
print("✅ Comprehensive Data Cleansing Pipeline initialized")

## 6. Execute Pipeline on Sample Data

In [ ]:
# Test the pipeline with key Federal Reserve series
print("🧪 Testing pipeline with sample Federal Reserve series")
print("=" * 60)

# Process key series for demonstration
sample_series = ['rates', 'inflation']
results = pipeline.process_all_series(sample_series)

print(f"\n📊 Processing Results:")
print(f"Successful series: {results['success']}")
print(f"Failed series: {results['failed']}")

## 7. Quality Assessment Report

In [ ]:
# Generate and display quality report
if pipeline.quality_reports:
    quality_report = pipeline.generate_quality_report()
    
    print("\n📋 Data Quality Assessment Report")
    print("=" * 60)
    display(quality_report)
    
    # Summary statistics
    print(f"\n📈 Quality Summary:")
    print(f"Total series processed: {len(quality_report)}")
    print(f"Good quality series: {len(quality_report[quality_report['Quality_Flag'] == 'GOOD'])}")
    print(f"Poor quality series: {len(quality_report[quality_report['Quality_Flag'] == 'POOR'])}")
    print(f"Average record count: {quality_report['Final_Records'].mean():.0f}")
    print(f"Average missing data %: {quality_report['Final_Missing_%'].mean():.2f}%")
else:
    print("⚠️ No quality reports available. Run the pipeline first.")

## 8. Cleaning Actions Log

In [ ]:
# Display cleaning actions log
cleaning_summary = pipeline.cleaner.get_cleaning_summary()

if not cleaning_summary.empty:
    print("\n🔧 Data Cleaning Actions Log")
    print("=" * 60)
    display(cleaning_summary[['timestamp', 'action', 'series', 'details']].tail(20))
    
    # Action summary
    action_counts = cleaning_summary['action'].value_counts()
    print(f"\n📊 Cleaning Actions Summary:")
    for action, count in action_counts.items():
        print(f"{action}: {count} actions")
else:
    print("⚠️ No cleaning actions recorded.")

## 9. Data Visualization

In [ ]:
# Visualize sample cleaned data
if pipeline.cleaned_datasets:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Sample Cleaned Federal Reserve Data', fontsize=16, fontweight='bold')
    
    plot_count = 0
    
    for series_id, df in list(pipeline.cleaned_datasets.items())[:4]:
        if df.empty:
            continue
            
        row = plot_count // 2
        col = plot_count % 2
        ax = axes[row, col]
        
        # Plot the time series
        ax.plot(df['date'], df[series_id], linewidth=2)
        ax.set_title(f'{series_id}\n({config.FRED_SERIES.get("rates", {}).get(series_id, series_id)})')
        ax.set_xlabel('Date')
        ax.set_ylabel('Value')
        ax.grid(True, alpha=0.3)
        
        # Rotate x-axis labels
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
        
        plot_count += 1
        
        if plot_count >= 4:
            break
    
    # Hide empty subplots
    for i in range(plot_count, 4):
        row = i // 2
        col = i % 2
        axes[row, col].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    print("✅ Visualization of cleaned data generated")
else:
    print("⚠️ No cleaned datasets available for visualization")

## 10. Export Cleaned Data

In [ ]:
# Export cleaned datasets
if pipeline.cleaned_datasets:
    print("💾 Exporting cleaned datasets...")
    
    # Export as CSV files
    pipeline.export_cleaned_data('csv', 'cleaned_fed_data')
    
    # Export quality report
    if pipeline.quality_reports:
        quality_report = pipeline.generate_quality_report()
        quality_report.to_csv('cleaned_fed_data/data_quality_report.csv', index=False)
        print("✅ Quality report exported")
    
    # Export cleaning log
    cleaning_log = pipeline.cleaner.get_cleaning_summary()
    if not cleaning_log.empty:
        cleaning_log.to_csv('cleaned_fed_data/cleaning_actions_log.csv', index=False)
        print("✅ Cleaning actions log exported")
    
    print(f"\n📁 All files exported to 'cleaned_fed_data' directory")
    print(f"📊 Total datasets: {len(pipeline.cleaned_datasets)}")
else:
    print("⚠️ No cleaned datasets to export")

## 11. Pipeline Summary and Recommendations

In [ ]:
print("\n" + "=" * 70)
print("📋 FEDERAL RESERVE DATA CLEANSING PIPELINE SUMMARY")
print("=" * 70)

print(f"\n🎯 Pipeline Configuration:")
print(f"• FRED API Key: {'✅ Configured' if config.FRED_API_KEY else '❌ Missing'}")
print(f"• Max Missing Data: {config.MAX_MISSING_PERCENTAGE * 100}%")
print(f"• Outlier Threshold: {config.OUTLIER_Z_SCORE_THRESHOLD} standard deviations")
print(f"• Date Range: {config.DEFAULT_START_DATE} to {config.DEFAULT_END_DATE}")

if pipeline.cleaned_datasets:
    print(f"\n📊 Processing Results:")
    print(f"• Total series processed: {len(pipeline.cleaned_datasets)}")
    print(f"• Average records per series: {np.mean([len(df) for df in pipeline.cleaned_datasets.values()]):.0f}")
    print(f"• Date range coverage: {min([df['date'].min() for df in pipeline.cleaned_datasets.values() if not df.empty])} to {max([df['date'].max() for df in pipeline.cleaned_datasets.values() if not df.empty])}")

if pipeline.quality_reports:
    quality_df = pipeline.generate_quality_report()
    good_quality = len(quality_df[quality_df['Quality_Flag'] == 'GOOD'])
    
    print(f"\n✅ Data Quality Assessment:")
    print(f"• Good quality series: {good_quality}/{len(quality_df)} ({good_quality/len(quality_df)*100:.1f}%)")
    print(f"• Average data completeness: {100 - quality_df['Final_Missing_%'].mean():.1f}%")

print(f"\n🔧 Cleaning Actions Performed:")
cleaning_log = pipeline.cleaner.get_cleaning_summary()
if not cleaning_log.empty:
    action_counts = cleaning_log['action'].value_counts()
    for action, count in action_counts.items():
        print(f"• {action.replace('_', ' ').title()}: {count} actions")
else:
    print("• No cleaning actions recorded")

print(f"\n💡 Recommendations:")
print(f"• Use cleaned datasets for Federal Reserve policy analysis")
print(f"• Monitor data quality flags for series reliability")
print(f"• Re-run pipeline monthly to incorporate new data")
print(f"• Review cleaning actions log for transparency")
print(f"• Consider custom cleaning rules for specific series")

print(f"\n✅ Pipeline execution completed successfully!")
print("=" * 70)